## Over-simplified FASTA Algorithm

#### BC205: Algorithms for Bioinformatics - Exercise III

In [1]:
#insert query and database file in the same folder

#parse genome file and store to dictionary 'seqid2seq'
seqid2seq={}
with open("target_file.fa")as fa_file:
    seqid = None
    
    for line in fa_file:
        line = line.strip()
      
        if line.startswith(">"):
            header = line[1:]
            fields = header.split()
            seqid = fields[0]
            seqid2seq[seqid] = ''
        else:
            seqid2seq[seqid] += line


In [2]:
#parse query file and store to string 'query'
query=''
with open("query.fa") as query_file:
    
    for line in query_file:
        line = line.strip()
      
        if line.startswith(">"):
            header = line[1:]
        else:
            query += line

In [ ]:
#test for query dictionary

#print(seqid2seq['YAL024C']) # example gene
#print(query)

## Parameters Input

In [4]:
# Parameters
k = 7  # size of k-mer
m = 20  # minimum size of matched diagonals (S[i])
g = 3  # maximum gap size before joining regions
score_threshold_fraction = 0.5  # percentage of similarity



## Finding k-mer positions

In [5]:
# create a dictionary to match each k-mer to its position

def find_kmer_matches(query_seq, target_seq, k): # insert query seq, target and k 
    matches = []  # (i, j) positions where query == target
    kmer_dict = {}

    # Index all k-mers in target
    for j in range(len(target_seq) - k + 1):
        kmer = target_seq[j:j+k]
        if kmer not in kmer_dict:
            kmer_dict[kmer] = [] #create list for each k-mer to append all different positions
        kmer_dict[kmer].append(j)

    # Check which k-mers in query can be found in target
    for i in range(len(query_seq) - k + 1):
        kmer = query_seq[i:i+k]
        if kmer in kmer_dict:
            for j in kmer_dict[kmer]:
                matches.append((i, j))  # matched query[i] and target[j] stored in tuple
    return matches

## Diagonal score for each query-target pair

In [6]:
# Function to group matches by diagonal
def group_by_diagonal(matches):
    diagonals = {}
    for i, j in matches:
        diag = i - j #diagonal ID of the match (same for all the matches that fall on the same diagonal)
        if diag not in diagonals:
            diagonals[diag] = []
        diagonals[diag].append((i, j))
    return diagonals

### Joint gaps 

In [7]:
# Function to join adjacent matches on same diagonal
def join_regions(diagonal_matches, g):
    regions = []
    sorted_matches = sorted(diagonal_matches, key=lambda x: x[0])  # sort by query position
    region = [sorted_matches[0]]

    #compare current match to previous one
    for i in range(1, len(sorted_matches)): 
        prev = region[-1]
        curr = sorted_matches[i]

        # choose for gap length
        if curr[0] - prev[0] <= g and curr[1] - prev[1] <= g:
            region.append(curr)
        else:
            regions.append(region)
            region = [curr]
    regions.append(region)
    return regions

## Compare sequences

In [12]:
#minimum score needed to accept a match as significant
min_score = int(len(query) * score_threshold_fraction)

# Track best match
best_score = 0
best_gene = ''
all_matches = []

# Compare query to each target
for gene_id, target_seq in seqid2seq.items():
    matches = find_kmer_matches(query, target_seq, k)
    diagonals = group_by_diagonal(matches)

    # Join diagonal regions and filter by size (m)
    final_regions = []
    for diag in diagonals:
        joined = join_regions(diagonals[diag], g)
        for region in joined:
            if len(region) * k >= m:  # each k-mer match gives k score
                final_regions.append(region)
 
    # Score = total length of matched regions
    score = sum(len(region) * k for region in final_regions)

    if score >= min_score:
        all_matches.append((gene_id, score))
        if score > best_score:
            best_score = score
            best_gene = gene_id

### Output

In [14]:
# Output results
print(f"Matched sequences (score ≥ 50% of query length, k-mer size: {k}):\n")
for gene, score in all_matches:
    print(f"{gene}: score = {score}")

print("\nBest matching sequence:")
print(f"{best_gene} with score = {best_score}")

Matched sequences (score ≥ 50% of query length, k-mer size: 7):

YGR142W: score = 1015
Q0080: score = 1015
Q0130: score = 1022
YMR021C: score = 12964

Best matching sequence:
YMR021C with score = 12964
